# Experiment 1 Dense Probe Training from Zip Archive (Weights & Biases edition)

**This notebook is a wandb-enabled copy of `notebooks/colab_exp1_zip_probe_training.ipynb`, restricted to the dense per-patch probes.**

It runs:

1. `scripts/train_all_dense_depth_probes.py` over `configs/exp1_dense.yaml`.
2. `scripts/train_all_dense_surface_normal_probes.py` over the same config.
3. `scripts/aggregate_exp1_results.py` + `scripts/make_exp1_figures.py` on the dense outputs only.

The original notebook also trains global CLS-token linear probes via `train_all_exp1_probes.py`; **those stages are intentionally removed here** so this notebook focuses on the dense sub-study.

What is added on top of the original behavior:

1. Each probe is tracked as a separate Weights & Biases run, with full hyperparameter / path metadata logged to `wandb.config`.
2. Every epoch row from `history.csv` is streamed to wandb so train/val curves, learning-rate schedules, and angular-error / SSI-L1 metrics are inspectable in the wandb UI.
3. Final test metrics are written to `wandb.summary` and the on-disk artifacts (`metrics.json`, `history.csv`, `predictions.csv`, optional `checkpoint.pt`) are uploaded as a wandb `Artifact`.
4. A separate wandb project is used for the aggregated figures/tables, so they stay decoupled from the per-probe training runs.
5. The training scripts run with `--epochs 50` by default (was ~10 effective with the original `EPOCHS=None` ceiling of 20 in the config plus early stopping). Early stopping with `patience=8` is still active, so most probes will not actually run 50 epochs.
6. A single-batch dense surface-normal smoke test for quick sanity checking before launching the full grid.

The training itself is **unchanged**: same configs (`configs/exp1_dense.yaml`), same probe heads, same losses, same optimizer, same metrics. WandB logging happens *after* each script finishes, by reading the per-probe outputs the script already writes to disk. Setting `USE_WANDB = False` reverts the notebook to the exact behavior of the original (minus the global probe stages).

**Original notebook:** `notebooks/colab_exp1_zip_probe_training.ipynb` (do not modify).

**Test-set warning:** the test split should be evaluated **once** at the end of a run. Do not iterate on hyperparameters using test metrics; use the validation curves instead.

Use this after `colab_exp1_zip_feature_extraction.ipynb` has copied dense patch-feature caches to Drive.

## What changed vs the original notebook

| Area | Original | This notebook |
| --- | --- | --- |
| Stages | global linear probes + dense depth + dense normals + aggregate/figures | **dense depth + dense normals + aggregate/figures only** |
| Probe training scripts | `!python scripts/train_all_*.py` | Same `train_all_dense_*` calls, unchanged. Optional override CLI flags exposed via `EPOCHS` / `BATCH_SIZE` / `LR` / `WEIGHT_DECAY` / `SEED` / `MODEL_NAME` / `LAYER` / `TEXTURE` knobs. |
| Max epochs | Config default (`20` in `exp1_dense.yaml`); ~10 effective with early stopping | `EPOCHS = 50` by default; early stopping (`patience=8`) still active. |
| Per-probe outputs (`metrics.json`, `history.csv`, `predictions.csv`, `checkpoint.pt`) | Saved on disk only | Saved on disk **and** mirrored into wandb project `probe3d-exp1-dense-probes` (curves + summary + artifact). |
| Per-epoch metrics | Saved to `history.csv` only | Streamed into wandb after each script run. |
| Figures + aggregate tables | Saved on disk | Same, plus uploaded to a **separate** wandb project `probe3d-exp1-dense-figures` as a `results` artifact. |
| Drive sync of outputs | `outputs/exp1_main*/` -> Drive | Only `outputs/exp1_dense/` + the wandb run directory tree are synced. |
| Smoke test | Not present | New optional dense surface-normal single-batch smoke test. |
| Compare against reference history | Not present | New optional cell that diffs new vs reference dense `metrics.json` for one selected probe. |

## 1. Mount Drive, unpack repo zip, configure paths

Set `ZIP_PATH` and `DRIVE_RESULTS` below to match where you uploaded the project zip and where you want results synced. Everything else is derived from the zip layout, so there are no machine-specific local paths to edit.

Dense patch-feature caches must already exist in Google Drive (produced by `colab_exp1_zip_feature_extraction.ipynb`). The cells below mount Drive and unpack the repo into Colab's local SSD.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

ZIP_PATH = Path('/content/drive/MyDrive/cv-project-exp1-rerender-colab.zip')
UNPACK_BASE = Path('/content/cv-project')
DRIVE_RESULTS = Path('/content/drive/MyDrive/cv-project-exp1-rerender-results')


def _find_exp1_root(base: Path) -> Path:
    """Zip archives often add one top-level folder; resolve to the tree that has exp1/."""
    marker = base / 'exp1' / 'features' / '__init__.py'
    if marker.is_file():
        return base
    for child in sorted(base.iterdir()):
        if child.is_dir() and (child / 'exp1' / 'features' / '__init__.py').is_file():
            return child
    raise FileNotFoundError(
        f'Could not find exp1 package under {base}. '
        'Re-zip the repo so it includes the full exp1/ directory.'
    )


assert ZIP_PATH.is_file(), f'Missing zip file: {ZIP_PATH}'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

!rm -rf /content/cv-project
!mkdir -p /content/cv-project
!unzip -q -o "{ZIP_PATH}" -d "{UNPACK_BASE}"

WORKDIR = _find_exp1_root(UNPACK_BASE)
os.environ['CV_PROJECT_ROOT'] = str(WORKDIR)
os.environ['PYTHONPATH'] = str(WORKDIR)
%cd {WORKDIR}

DENSE_DIR = WORKDIR / 'data' / 'exp1_dense'
FEATURE_DIR = DENSE_DIR / 'features'
MANIFEST_PATH = DENSE_DIR / 'manifests' / 'render_valid.parquet'
OUTPUT_DIR = WORKDIR / 'outputs' / 'exp1_dense'
WANDB_RUN_DIR = WORKDIR / 'outputs' / 'colab_probe_wandb_runs'
WANDB_RUN_DIR.mkdir(parents=True, exist_ok=True)

print('WORKDIR        :', WORKDIR)
print('DENSE_DIR      :', DENSE_DIR)
print('FEATURE_DIR    :', FEATURE_DIR)
print('MANIFEST_PATH  :', MANIFEST_PATH)
print('OUTPUT_DIR     :', OUTPUT_DIR)
print('WANDB_RUN_DIR  :', WANDB_RUN_DIR)
!nvidia-smi

In [ ]:
import sys

!{sys.executable} -m pip install -q -r requirements.txt
!{sys.executable} -m pip install -q pyarrow matplotlib seaborn scikit-learn
# wandb is the only added dependency vs the original notebook.
!{sys.executable} -m pip install -q wandb

## 2. Run configuration

Most users only need to set `USE_WANDB`, `WANDB_PROBE_PROJECT`, `WANDB_FIGURES_PROJECT`, and (optionally) `WANDB_ENTITY`. All hyperparameter overrides default to sensible values for the dense study; setting any of the `MODEL_NAME` / `LAYER` / `TEXTURE` filters restricts which probes are trained.

**Choosing a model / layer / texture:**

- `MODEL_NAME`: subset of `models.dense_enabled` from the config (default `["clip_vit_b16", "dinov2_vit_b"]`).
- `LAYER`: subset of `models.dense_layers` (default `["final", "layer8"]`).
- `TEXTURE`: optional explicit texture filter. When unset, the config's within-texture grid (`photorealistic`, `flat`, `random_noise`) is used. Cross-texture is **not** enabled for dense probes in this config.

**Tasks:** this notebook always trains both dense tasks (`dense_depth_patches` and `dense_surface_normal_patches`). Disable a stage with `RUN_DENSE_DEPTH_PROBES` / `RUN_DENSE_NORMAL_PROBES` if needed.

**Epochs:** the dense config defaults `epochs: 20`. We bump the ceiling to `EPOCHS = 50` so the trainer has more room to converge. Early stopping (`patience=8`, `min_delta=0`) is still active, so most probes will stop well before 50.

**Reading wandb training curves:** train_loss should drop monotonically; validation angular-error / SSI-L1 should plateau as the probe converges. Test metrics are logged once at the end of each probe and shown in the run summary; do not use them for hyperparameter selection.

In [ ]:
USE_WANDB = True
WANDB_PROBE_PROJECT = 'probe3d-exp1-dense-probes'
WANDB_FIGURES_PROJECT = 'probe3d-exp1-dense-figures'
WANDB_ENTITY = None
WANDB_RUN_GROUP = 'colab_zip_exp1_dense'
WANDB_TAGS = ['exp1', 'colab', 'dense']

DENSE_CONFIG = WORKDIR / 'configs' / 'exp1_dense.yaml'

MODEL_NAME = []
LAYER = []
TEXTURE = []

SEED = None
EPOCHS = 50
BATCH_SIZE = 256
LR = None
WEIGHT_DECAY = None

RUN_DENSE_DEPTH_PROBES = True
RUN_DENSE_NORMAL_PROBES = True
RUN_AGGREGATE_AND_FIGURES = True
RUN_SMOKE_TEST = True

INCLUDE_CHECKPOINTS_IN_ARTIFACT = False

assert DENSE_CONFIG.is_file(), f'Missing config: {DENSE_CONFIG}'

print('USE_WANDB              :', USE_WANDB)
print('WANDB_PROBE_PROJECT    :', WANDB_PROBE_PROJECT)
print('WANDB_FIGURES_PROJECT  :', WANDB_FIGURES_PROJECT)
print('DENSE_CONFIG           :', DENSE_CONFIG)
print('EPOCHS                 :', EPOCHS)

In [ ]:
import wandb

if USE_WANDB:
    wandb.login()
    print('wandb version:', wandb.__version__)
else:
    print('USE_WANDB=False; skipping wandb.login(). All training cells will still run, '
          'just without wandb tracking.')

## 3. Normalize manifest paths (preserved from original notebook)

Older zips contain absolute Mac paths under `/Users/jerry/cv-project/...`. Rewriting them to repository-relative paths lets the dense probe scripts resolve depth / normal / mask buffers against the Colab project root. This cell is idempotent: re-running it is safe.

In [ ]:
import pandas as pd
from pathlib import Path

PATH_COLUMNS = (
    'raw_mesh_path',
    'normalized_mesh_path',
    'rgb_path',
    'depth_path',
    'normal_path',
    'mask_path',
    'mesh_import_path',
    'random_texture_path',
)
MANIFESTS = [
    'data/exp1_dense/manifests/render_valid.parquet',
    'data/exp1_dense/manifests/render_qc.parquet',
]
ANCHORS = ('data', 'configs', 'exp1', 'scripts', 'src')


def _relativize(value):
    if value is None:
        return value
    text = str(value)
    if not text or text.lower() == 'nan':
        return text
    path = Path(text)
    if not path.is_absolute():
        return text
    parts = path.parts
    for anchor in ANCHORS:
        if anchor in parts:
            return str(Path(*parts[parts.index(anchor):]))
    return text


for rel in MANIFESTS:
    p = WORKDIR / rel
    if not p.is_file():
        print(f'skipped (missing): {rel}')
        continue
    df = pd.read_parquet(p)
    changed_cols = []
    for col in PATH_COLUMNS:
        if col not in df.columns:
            continue
        before = df[col].astype('object')
        after = before.map(_relativize)
        if not (before == after).all():
            df[col] = after
            changed_cols.append(col)
    if changed_cols:
        df.to_parquet(p, index=False)
        print(f'rewrote {rel}: {changed_cols}')
    else:
        print(f'already relative: {rel}')

## 4. Restore dense feature cache from Drive

Probe training reads cached frozen patch-feature `.npz` files written by `colab_exp1_zip_feature_extraction.ipynb`. If they are missing on Drive, run that notebook first; this notebook intentionally does not re-extract features (the backbones must remain frozen and re-extraction would be wasteful).

In [ ]:
import shutil

rel = 'data/exp1_dense/features'
src = DRIVE_RESULTS / rel
dst = WORKDIR / rel
if not src.exists():
    raise FileNotFoundError(
        f'Missing dense feature cache directory on Drive: {src}\n'
        'Generate it with notebooks/colab_exp1_zip_feature_extraction.ipynb '
        'before running this dense probe-training notebook.'
    )
if dst.exists():
    shutil.rmtree(dst)
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst)
print(f'Restored {src} -> {dst}')

## 5. WandB logging helpers

These helpers walk a probe output directory (e.g. `outputs/exp1_dense/probes/<model>/<layer>/<task>/<texture_job>/`) and stream the per-probe artifacts into a separate wandb run.

Each probe directory contains `metrics.json`, `history.csv`, `predictions.csv`, and `checkpoint.pt` exactly as the original notebook produces them. The helpers below only **read** those files – they do not change the training pipeline.

In [ ]:
import json
import math
import sys
from pathlib import Path

import pandas as pd

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

from exp1.config import load_exp1_config, resolve_path


def _yaml_safe(value):
    """Convert OmegaConf / numpy scalars into wandb-friendly Python primitives."""
    if value is None or isinstance(value, (bool, int, float, str)):
        return value
    if isinstance(value, (list, tuple)):
        return [_yaml_safe(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _yaml_safe(v) for k, v in value.items()}
    return str(value)


def _flatten_metrics(metrics_by_split):
    flat = {}
    for split_name, split_metrics in metrics_by_split.items():
        if not isinstance(split_metrics, dict):
            continue
        for key, value in split_metrics.items():
            try:
                flat[f'{split_name}/{key}'] = float(value)
            except (TypeError, ValueError):
                continue
    return flat


def _row_to_log_dict(row):
    out = {}
    for key, value in row.items():
        if key == 'epoch':
            continue
        try:
            f = float(value)
        except (TypeError, ValueError):
            continue
        if math.isnan(f):
            continue
        out[key] = f
    return out


def _per_probe_run_name(model, layer, task, texture_job, *, prefix):
    return f'{prefix}/{model}/{layer}/{task}/{texture_job}'


def log_probe_outputs_to_wandb(
    probes_root,
    *,
    config_path,
    project,
    entity=None,
    group=None,
    tags=None,
    name_prefix='dense',
    extra_config=None,
    include_checkpoints=False,
    only_models=None,
    only_layers=None,
    only_tasks=None,
):
    """Walk a probe output tree and create one wandb run per saved probe.

    The directory layout matches what scripts/train_all_*.py write:
    ``probes_root/<model>/<layer>/<task>/<texture_job>/{history.csv,metrics.json,...}``.
    """
    probes_root = Path(probes_root)
    if not probes_root.is_dir():
        print(f'log_probe_outputs_to_wandb: nothing to log at {probes_root}')
        return []
    if not USE_WANDB:
        print('USE_WANDB=False; skipping wandb upload of', probes_root)
        return []
    cfg = load_exp1_config(Path(config_path))
    project_root = Path(str(cfg.paths.project_root)).expanduser().resolve()
    feature_dir = resolve_path(project_root, str(cfg.paths.feature_dir))
    manifest_resolved = resolve_path(project_root, str(cfg.paths.valid_render_manifest))
    base_extra = dict(extra_config or {})
    base_extra.setdefault('feature_dir', str(feature_dir))
    base_extra.setdefault('manifest_path', str(manifest_resolved))
    base_extra.setdefault('output_dir', str(probes_root))
    base_extra.setdefault('config_path', str(config_path))
    only_models = set(only_models or [])
    only_layers = set(only_layers or [])
    only_tasks = set(only_tasks or [])

    runs = []
    metrics_files = sorted(probes_root.rglob('metrics.json'))
    if not metrics_files:
        print(f'log_probe_outputs_to_wandb: no metrics.json found under {probes_root}')
        return runs
    for metrics_path in metrics_files:
        probe_dir = metrics_path.parent
        try:
            relative = probe_dir.relative_to(probes_root)
        except ValueError:
            continue
        parts = relative.parts
        if len(parts) != 4:
            continue
        model, layer, task, texture_job = parts
        if only_models and model not in only_models:
            continue
        if only_layers and layer not in only_layers:
            continue
        if only_tasks and task not in only_tasks:
            continue

        with metrics_path.open('r', encoding='utf-8') as fh:
            metrics_blob = json.load(fh)
        metrics_by_split = metrics_blob.get('metrics', {}) or {}
        metadata = metrics_blob.get('metadata', {}) or {}

        wandb_config = {
            'model_name': model,
            'layer': layer,
            'task': task,
            'texture_job': texture_job,
            'texture_condition': metadata.get('texture_condition'),
            'train_texture_condition': metadata.get('train_texture_condition'),
            'eval_texture_condition': metadata.get('eval_texture_condition'),
            'feature_cache': metadata.get('feature_cache') or metadata.get('patch_cache'),
            'label_path': metadata.get('label_path'),
            'manifest_path': metadata.get('manifest_path', str(manifest_resolved)),
            'num_rows_by_split': metadata.get('num_rows_by_split'),
            'probe_dir': str(probe_dir.relative_to(WORKDIR)),
        }
        wandb_config.update(_yaml_safe(base_extra))

        checkpoint_path = probe_dir / 'checkpoint.pt'
        if checkpoint_path.is_file():
            try:
                import torch

                ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
                train_cfg = _yaml_safe(ckpt.get('train_config', {})) or {}
                head_cfg = _yaml_safe(ckpt.get('head_config', {})) or {}
                for key in (
                    'lr', 'batch_size', 'weight_decay', 'epochs', 'seed',
                    'use_layernorm', 'scheduler', 'warmup_epochs', 'monitor',
                    'feature_mode', 'target_mode', 'min_valid_fraction_per_patch',
                    'depth_statistic', 'early_stopping', 'patience',
                    'min_delta', 'surface_normal_loss',
                ):
                    if key in train_cfg:
                        wandb_config[key] = train_cfg[key]
                if head_cfg:
                    wandb_config['head_config'] = head_cfg
            except Exception as exc:  # noqa: BLE001
                print(f'  warn: could not parse {checkpoint_path}: {exc}')

        run_name = _per_probe_run_name(model, layer, task, texture_job, prefix=name_prefix)
        per_run_dir = WANDB_RUN_DIR / run_name.replace('/', '__')
        per_run_dir.mkdir(parents=True, exist_ok=True)

        run = wandb.init(
            project=project,
            entity=entity,
            group=group,
            tags=tags,
            name=run_name,
            config=wandb_config,
            reinit=True,
            dir=str(per_run_dir),
        )
        runs.append(run)
        try:
            history_path = probe_dir / 'history.csv'
            if history_path.is_file():
                history_df = pd.read_csv(history_path)
                for _, row in history_df.iterrows():
                    if 'epoch' in row and not pd.isna(row['epoch']):
                        epoch = int(row['epoch'])
                    else:
                        epoch = int(row.name) + 1
                    payload = _row_to_log_dict(row)
                    if payload:
                        wandb.log(payload, step=epoch)
                wandb.log({'epochs_logged': int(len(history_df))})
            else:
                print(f'  warn: no history.csv in {probe_dir}')

            summary_metrics = _flatten_metrics(metrics_by_split)
            for key, value in summary_metrics.items():
                wandb.summary[f'final/{key}'] = value
            wandb.summary['final/test_split_present'] = 'test' in metrics_by_split

            artifact_files = ['metrics.json', 'history.csv', 'predictions.csv']
            if include_checkpoints:
                artifact_files.append('checkpoint.pt')
            artifact = wandb.Artifact(
                name=run_name.replace('/', '__'),
                type='probe_outputs',
                metadata=_yaml_safe(metadata),
            )
            for filename in artifact_files:
                file_path = probe_dir / filename
                if file_path.is_file():
                    artifact.add_file(str(file_path), name=filename)
            for extra in probe_dir.glob('predictions_*.npz'):
                artifact.add_file(str(extra), name=extra.name)
            wandb.log_artifact(artifact)
        finally:
            run.finish()
    print(f'Logged {len(runs)} probe(s) from {probes_root} to wandb project {project!r}')
    return runs

## 6. Smoke test (optional)

Loads a small batch of cached patch features for one (model, layer) combination, builds the dense surface-normal target tensor from the manifest's depth/mask buffers, runs one forward + backward pass through a fresh `DenseSurfaceNormalHead`, and prints tensor shapes. This is a fast (\~tens of seconds, dominated by the first-time patch-cache load) sanity check that patch caches, manifest path columns, and rendered normal buffers all join correctly **before** kicking off the full probe grid.

In [ ]:
if RUN_SMOKE_TEST:
    import torch

    from exp1.features.storage import patch_feature_cache_path
    from exp1.metadata.manifest import load_manifest
    from exp1.probes.dense_surface_normals import (
        DenseSurfaceNormalHead,
        DenseSurfaceNormalHeadConfig,
        dense_surface_normal_loss,
    )
    from exp1.tasks.dense_surface_normals import Exp1DenseSurfaceNormalDataset

    cfg = load_exp1_config(DENSE_CONFIG)
    project_root = Path(str(cfg.paths.project_root)).expanduser().resolve()
    feat_dir = resolve_path(project_root, str(cfg.paths.feature_dir))
    manifest_path = resolve_path(project_root, str(cfg.paths.valid_render_manifest))

    dense_enabled = cfg.models.get('dense_enabled') or cfg.models.enabled
    dense_layers = cfg.models.get('dense_layers') or cfg.models.layers
    smoke_model = (MODEL_NAME[0] if MODEL_NAME else str(dense_enabled[0]))
    smoke_layer = (LAYER[0] if LAYER else str(dense_layers[0]))
    patch_cache = patch_feature_cache_path(
        feat_dir, model_name=smoke_model, layer_name=smoke_layer
    )
    assert Path(patch_cache).is_file(), (
        f'Missing patch cache: {patch_cache}. '
        'Run colab_exp1_zip_feature_extraction.ipynb first.'
    )
    manifest = load_manifest(manifest_path, validate=False)

    dataset = Exp1DenseSurfaceNormalDataset(
        patch_cache,
        manifest=manifest,
        split='train',
        project_root=project_root,
        feature_mode='patch',
    )
    arrays = dataset.materialize_arrays()
    print(
        f'Smoke test: model={smoke_model} layer={smoke_layer} '
        f'n_train={arrays["features"].shape[0]} '
        f'patch_grid={dataset.patch_grid_shape} '
        f'feature_dim={dataset.feature_dim}'
    )
    if arrays['features'].shape[0] == 0:
        print('Smoke test skipped: no train rows after manifest/join filtering.')
    else:
        n_smoke = min(8, arrays['features'].shape[0])
        x = torch.from_numpy(arrays['features'][:n_smoke]).float()
        y = torch.from_numpy(arrays['targets'][:n_smoke]).float()
        m = torch.from_numpy(arrays['valid'][:n_smoke]).bool()

        head = DenseSurfaceNormalHead(
            DenseSurfaceNormalHeadConfig(feature_dim=int(dataset.feature_dim))
        )
        optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
        optimizer.zero_grad(set_to_none=True)
        pred = head(x)
        loss = dense_surface_normal_loss(pred, y, m)
        loss.backward()
        optimizer.step()

        print('  features  :', tuple(x.shape))
        print('  targets   :', tuple(y.shape))
        print('  valid_mask:', tuple(m.shape))
        print('  pred      :', tuple(pred.shape))
        print('  loss      :', float(loss.item()))
        print('Smoke test OK')
else:
    print('RUN_SMOKE_TEST=False; skipping smoke test.')

## 7. Dense patch-depth probes

Runs `scripts/train_all_dense_depth_probes.py` against `configs/exp1_dense.yaml`, with `--epochs 50` (overriding the config default of 20). After the script finishes, the per-probe outputs are mirrored into the wandb project `WANDB_PROBE_PROJECT`.

Dense-depth metrics live under names such as `train_ssi_l1_mean`, `val_ssi_l1_mean`, `train_pearson_r_mean`, etc. The wandb runs additionally include the `lr` column from `history.csv` so the cosine-warmup schedule is visible.

In [ ]:
if RUN_DENSE_DEPTH_PROBES:
    cmd = [
        'python', 'scripts/train_all_dense_depth_probes.py',
        '--config', str(DENSE_CONFIG),
        '--device', 'cuda',
    ]
    if BATCH_SIZE is not None:
        cmd += ['--batch-size', str(int(BATCH_SIZE))]
    if EPOCHS is not None:
        cmd += ['--epochs', str(int(EPOCHS))]
    if LR is not None:
        cmd += ['--lr', str(float(LR))]
    if WEIGHT_DECAY is not None:
        cmd += ['--weight-decay', str(float(WEIGHT_DECAY))]
    if SEED is not None:
        cmd += ['--seed', str(int(SEED))]
    if MODEL_NAME:
        cmd += ['--models', *MODEL_NAME]
    if LAYER:
        cmd += ['--layers', *LAYER]
    if TEXTURE:
        cmd += ['--texture-condition', *TEXTURE]
    print('+', ' '.join(cmd))
    !PYTHONPATH=. {' '.join(cmd)}
else:
    print('RUN_DENSE_DEPTH_PROBES=False; skipping dense depth probe training.')

In [ ]:
if RUN_DENSE_DEPTH_PROBES and USE_WANDB:
    log_probe_outputs_to_wandb(
        OUTPUT_DIR / 'probes',
        config_path=DENSE_CONFIG,
        project=WANDB_PROBE_PROJECT,
        entity=WANDB_ENTITY,
        group=WANDB_RUN_GROUP,
        tags=WANDB_TAGS + ['stage:dense_depth'],
        name_prefix='dense_depth',
        extra_config={
            'config_name': DENSE_CONFIG.name,
            'stage': 'dense_depth',
            'cli_overrides': {
                'epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
                'lr': LR,
                'weight_decay': WEIGHT_DECAY,
                'seed': SEED,
                'models': MODEL_NAME,
                'layers': LAYER,
                'texture': TEXTURE,
            },
        },
        include_checkpoints=INCLUDE_CHECKPOINTS_IN_ARTIFACT,
        only_models=MODEL_NAME,
        only_layers=LAYER,
        only_tasks={'dense_depth_patches'},
    )
elif RUN_DENSE_DEPTH_PROBES:
    print('USE_WANDB=False; skipping wandb upload of dense depth probes.')

## 8. Dense patch surface-normal probes

Final training stage: dense per-patch surface-normal probes. WandB runs report angular-error metrics (`train_angular_error_deg_mean`, `val_angular_error_deg_mean`) per epoch and as final summary.

In [ ]:
if RUN_DENSE_NORMAL_PROBES:
    cmd = [
        'python', 'scripts/train_all_dense_surface_normal_probes.py',
        '--config', str(DENSE_CONFIG),
        '--device', 'cuda',
    ]
    if BATCH_SIZE is not None:
        cmd += ['--batch-size', str(int(BATCH_SIZE))]
    if EPOCHS is not None:
        cmd += ['--epochs', str(int(EPOCHS))]
    if LR is not None:
        cmd += ['--lr', str(float(LR))]
    if WEIGHT_DECAY is not None:
        cmd += ['--weight-decay', str(float(WEIGHT_DECAY))]
    if SEED is not None:
        cmd += ['--seed', str(int(SEED))]
    if MODEL_NAME:
        cmd += ['--models', *MODEL_NAME]
    if LAYER:
        cmd += ['--layers', *LAYER]
    if TEXTURE:
        cmd += ['--texture-condition', *TEXTURE]
    print('+', ' '.join(cmd))
    !PYTHONPATH=. {' '.join(cmd)}
else:
    print('RUN_DENSE_NORMAL_PROBES=False; skipping dense surface-normal probe training.')

In [ ]:
if RUN_DENSE_NORMAL_PROBES and USE_WANDB:
    log_probe_outputs_to_wandb(
        OUTPUT_DIR / 'probes',
        config_path=DENSE_CONFIG,
        project=WANDB_PROBE_PROJECT,
        entity=WANDB_ENTITY,
        group=WANDB_RUN_GROUP,
        tags=WANDB_TAGS + ['stage:dense_surface_normal'],
        name_prefix='dense_normal',
        extra_config={
            'config_name': DENSE_CONFIG.name,
            'stage': 'dense_surface_normal',
            'cli_overrides': {
                'epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
                'lr': LR,
                'weight_decay': WEIGHT_DECAY,
                'seed': SEED,
                'models': MODEL_NAME,
                'layers': LAYER,
                'texture': TEXTURE,
            },
        },
        include_checkpoints=INCLUDE_CHECKPOINTS_IN_ARTIFACT,
        only_models=MODEL_NAME,
        only_layers=LAYER,
        only_tasks={'dense_surface_normal_patches'},
    )
elif RUN_DENSE_NORMAL_PROBES:
    print('USE_WANDB=False; skipping wandb upload of dense surface-normal probes.')

## 9. Aggregate dense results and regenerate figures

Same calls as the original notebook, but only for the dense config (`configs/exp1_dense.yaml`). After they finish, this notebook also creates a single `results_summary` wandb run in the **separate** `WANDB_FIGURES_PROJECT` that uploads the final aggregated CSV tables and figure PNGs as a wandb artifact for easy sharing. Keeping figures in a different project keeps the per-probe training runs uncluttered.

In [ ]:
if RUN_AGGREGATE_AND_FIGURES:
    !PYTHONPATH=. python scripts/aggregate_exp1_results.py --config configs/exp1_dense.yaml
    !PYTHONPATH=. python scripts/make_exp1_figures.py --config configs/exp1_dense.yaml
else:
    print('RUN_AGGREGATE_AND_FIGURES=False; skipping aggregation + figures.')

In [ ]:
if RUN_AGGREGATE_AND_FIGURES and USE_WANDB:
    summary_dir = WANDB_RUN_DIR / 'summary'
    summary_dir.mkdir(parents=True, exist_ok=True)
    summary_run = wandb.init(
        project=WANDB_FIGURES_PROJECT,
        entity=WANDB_ENTITY,
        group=WANDB_RUN_GROUP,
        tags=WANDB_TAGS + ['stage:summary'],
        name='results_summary',
        reinit=True,
        dir=str(summary_dir),
        config={
            'dense_config': str(DENSE_CONFIG),
            'probe_project': WANDB_PROBE_PROJECT,
        },
    )
    try:
        artifact = wandb.Artifact('exp1_dense_results_summary', type='results')
        stage_outputs = OUTPUT_DIR
        results_dir = stage_outputs / 'results'
        figures_dir = stage_outputs / 'figures'
        if results_dir.is_dir():
            for csv_path in sorted(results_dir.glob('*.csv')):
                artifact.add_file(
                    str(csv_path),
                    name=str(csv_path.relative_to(WORKDIR)),
                )
        if figures_dir.is_dir():
            for fig_path in sorted(figures_dir.rglob('*.png')):
                artifact.add_file(
                    str(fig_path),
                    name=str(fig_path.relative_to(WORKDIR)),
                )
        wandb.log_artifact(artifact)
    finally:
        summary_run.finish()
elif RUN_AGGREGATE_AND_FIGURES:
    print('USE_WANDB=False; skipping wandb upload of aggregated results.')

## 10. Copy outputs back to Drive

Same final sync as the original notebook (restricted to dense outputs), plus the wandb run-directory tree under `outputs/colab_probe_wandb_runs/<run_name>/` so wandb can resume / re-upload runs from another Colab session if needed.

In [ ]:
import shutil

for rel in ['outputs/exp1_dense', 'outputs/colab_probe_wandb_runs']:
    src = WORKDIR / rel
    dst = DRIVE_RESULTS / rel
    if dst.exists():
        shutil.rmtree(dst)
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst)
        print(f'Copied {src} -> {dst}')
    else:
        print(f'Skipped missing {src}')

## 11. Optional: compare new dense probe metrics against a reference run

If you previously ran `colab_exp1_zip_probe_training.ipynb` (or any other source) and saved its outputs under `DRIVE_RESULTS/outputs/exp1_dense_reference/probes/...`, this cell prints a side-by-side comparison of the final test metrics for one selected probe between the new run and the reference. Useful for spotting regressions caused by environment / dependency drift, or for confirming that the new 50-epoch ceiling produced better metrics than the previous run.

Edit `COMPARE_PROBE` to point at any subdirectory (relative to `outputs/exp1_dense/probes/`).

In [ ]:
import json
from pathlib import Path

REFERENCE_DRIVE = DRIVE_RESULTS / 'outputs' / 'exp1_dense_reference'
COMPARE_PROBE = 'clip_vit_b16/final/dense_surface_normal_patches/within_photorealistic'

new_metrics_path = OUTPUT_DIR / 'probes' / COMPARE_PROBE / 'metrics.json'
ref_metrics_path = REFERENCE_DRIVE / 'probes' / COMPARE_PROBE / 'metrics.json'


def _load(path):
    if not Path(path).is_file():
        return None
    with open(path, 'r', encoding='utf-8') as fh:
        blob = json.load(fh)
    return blob.get('metrics', {})


new_metrics = _load(new_metrics_path)
ref_metrics = _load(ref_metrics_path)
if new_metrics is None:
    print(f'No new metrics at {new_metrics_path}; run the dense probe training first.')
elif ref_metrics is None:
    print(f'No reference metrics at {ref_metrics_path}; nothing to compare against.')
else:
    print(f'Comparing test metrics for {COMPARE_PROBE}:')
    new_test = new_metrics.get('test', {})
    ref_test = ref_metrics.get('test', {})
    keys = sorted(set(new_test) | set(ref_test))
    print(f'{"metric":40s} {"new":>14s} {"reference":>14s} {"delta":>14s}')
    for key in keys:
        n = new_test.get(key)
        r = ref_test.get(key)
        if n is None or r is None:
            delta = 'n/a'
        else:
            try:
                delta = f'{float(n) - float(r):+.4f}'
            except (TypeError, ValueError):
                delta = 'n/a'
        n_str = f'{n:.4f}' if isinstance(n, (int, float)) else str(n)
        r_str = f'{r:.4f}' if isinstance(r, (int, float)) else str(r)
        print(f'{key:40s} {n_str:>14s} {r_str:>14s} {delta:>14s}')